# LangChain: Tools, Chains, Memory & Agent

Rebuilding Day 1's raw-Python agent using **LangChain**, then going further: chaining prompts with LCEL, managing conversation history, using a tool that reads from a real local data source, and producing structured output.

We use **Groq** as the model provider through LangChain's `langchain-groq` integration. The focus of this notebook is on understanding how LangChain provides abstractions for tools, agents, conversation history, chains, structured output, and error handling, while comparing these abstractions with the raw-Python implementation from Day 1.


In [1]:
!pip install -U "langchain==0.3.27" "langchain-core==0.3.72" "langchain-groq"

INFO: pip is looking at multiple versions of langchain-groq to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of langchain-text-splitters to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.8/442.8 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.2 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.9
    Uninstalling langchain-core-1.4.9:
      Successfully uninstalled langchain-core-1.4.9
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.13
    Uninstalling langchain-1.3.13:
      Successfully uninstalled langchain-1.3.13
ERROR: pip's dependency resolver does not currently take into account all the packages that are inst

In [2]:
import os
import json
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

from langchain_groq import ChatGroq
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

llm = ChatGroq(model="llama-3.3-70b-versatile", api_key=api_key, temperature=0)
print("Setup complete and model initialized!")

Setup complete and model initialized!


## Task 1: LangChain Setup & Core Concepts

### Mapping LangChain Concepts to Day 1's Raw-Python Equivalents

| Day 1 — Raw Python                                                               | LangChain Equivalent                                                                                                                            |     |                                                                                                                                    |
| -------------------------------------------------------------------------------- | ----------------------------------------------------------------------------------------------------------------------------------------------- | --- | ---------------------------------------------------------------------------------------------------------------------------------- |
| `client = OpenAI(base_url=..., api_key=...)`                                     | `ChatGroq(model=..., api_key=...)` — a **chat-model wrapper** that provides a standard LangChain interface for interacting with the Groq model. |     |                                                                                                                                    |
| Hand-written `{"type": "function", "function": {...}}` tool schema               | `@tool` decorator — LangChain uses the function name, docstring, type hints, and signature to generate the tool definition/schema.              |     |                                                                                                                                    |
| Our own `while` loop (`run_agent`) implementing the Reason → Act → Observe cycle | `AgentExecutor` — manages the agent execution and tool-calling loop so we do not have to implement the orchestration manually.                  |     |                                                                                                                                    |
| Our manually maintained `messages` list                                          | `RunnableWithMessageHistory` — manages conversation history and makes previous messages available to later invocations.                         |     |                                                                                                                                    |
| *(No direct equivalent in Day 1)*                                                | **LCEL** (`prompt                                                                                                                               | llm | parser`) — a way to compose fixed processing steps into a sequential pipeline, similar to the workflow concept discussed in Day 1. |

### A Basic LCEL Pipeline


In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template("Answer in one short sentence: {question}")
chain = prompt | llm | StrOutputParser()

result = chain.invoke({"question": "What is LangChain, in plain terms?"})
print(result)


LangChain is an open-source framework that helps developers build applications using large language models like chatbots and AI assistants.


### What is the pipe (`|`) syntax doing under the hood?

Each component in the LCEL chain implements LangChain's `Runnable` interface, allowing its output to be passed to the next component. The `|` operator composes these components into a `RunnableSequence`, so the input flows from the prompt to the language model and then to the output parser.

Conceptually, `prompt | llm | parser` represents the same sequence of operations as calling the components one after another: the prompt processes the input, the model receives the resulting prompt and generates a response, and the parser converts that response into the desired output format. This is similar to Day 1's workflow concept because the steps are executed in a fixed order, but LCEL expresses the workflow declaratively using the pipe operator rather than requiring us to write the orchestration manually.


## Task 2: Define & Register Tools

Two tools were reused from Day 1: `calculator` and `get_weather`. A third tool, `get_product_price`, was added using LangChain's `@tool` decorator. Unlike the demo weather tool, `get_product_price` reads product prices and specifications from a real local JSON data source, `products.json`.

The product database is used again in Task 4 for the three-turn memory scenario: finding the price of Laptop A, comparing it with Laptop B, and making a recommendation for a budget-conscious client. This allows the same real data source to demonstrate both tool usage and conversation memory.


In [4]:
# --- Set up a small local JSON "database" of products ------------------------------
_PRODUCTS_DB_PATH = "products.json"

_PRODUCTS = {
    "laptop a": {"price": 899.0, "specs": "16GB RAM, 512GB SSD, mid-range CPU"},
    "laptop b": {"price": 459.0, "specs": "8GB RAM, 256GB SSD, entry-level CPU"},
    "laptop c": {"price": 1299.0, "specs": "32GB RAM, 1TB SSD, high-end CPU"},
}

with open(_PRODUCTS_DB_PATH, "w") as f:
    json.dump(_PRODUCTS, f, indent=2)

print(f"Wrote {_PRODUCTS_DB_PATH} with {len(_PRODUCTS)} products.")


Wrote products.json with 3 products.


In [5]:
from langchain.tools import tool
import json
import ast
import operator as op

In [6]:
# -------------------------------------------------------------------
# Configuration & Demo Databases
# -------------------------------------------------------------------

_PRODUCTS_DB_PATH = "products.json"

_FAKE_WEATHER_DB = {
    "lahore": {"temp_c": 41, "condition": "sunny"},
    "karachi": {"temp_c": 34, "condition": "humid"},
    "islamabad": {"temp_c": 33, "condition": "cloudy"},
}

_ALLOWED_OPERATORS = {
    ast.Add: op.add,
    ast.Sub: op.sub,
    ast.Mult: op.mul,
    ast.Div: op.truediv,
    ast.Pow: op.pow,
    ast.USub: op.neg,
    ast.UAdd: op.pos,
}


# -------------------------------------------------------------------
# Safe Arithmetic Evaluator
# -------------------------------------------------------------------

def evaluate_expression(expression: str):
    """Evaluate a restricted arithmetic expression without using eval()."""

    tree = ast.parse(expression, mode="eval")

    def evaluate(node):
        # Numeric constants
        if isinstance(node, ast.Constant) and isinstance(
            node.value, (int, float)
        ):
            return node.value

        # Binary operations: +, -, *, /, **
        if isinstance(node, ast.BinOp):
            operator = _ALLOWED_OPERATORS.get(type(node.op))

            if operator is None:
                raise ValueError("Operator not allowed")

            left = evaluate(node.left)
            right = evaluate(node.right)

            return operator(left, right)

        # Unary operations: +5, -5
        if isinstance(node, ast.UnaryOp):
            operator = _ALLOWED_OPERATORS.get(type(node.op))

            if operator is None:
                raise ValueError("Unary operator not allowed")

            return operator(evaluate(node.operand))

        raise ValueError(
            "Only basic arithmetic expressions are allowed"
        )

    return evaluate(tree.body)


# -------------------------------------------------------------------
# Tools Definitions
# -------------------------------------------------------------------

@tool
def calculator(expression: str) -> str:
    """Evaluates a basic arithmetic expression and returns the numeric result.
    Use this for calculations, sums, or comparisons of numbers.
    Supports +, -, *, /, **, unary +/-, and parentheses.
    Example input: '(4 + 5) * 2' or '2 ** 3'.
    """
    try:
        result = evaluate_expression(expression)
        return str(result)

    except ZeroDivisionError:
        return f"ERROR: division by zero in expression '{expression}'"

    except (SyntaxError, ValueError, TypeError):
        return f"ERROR: invalid arithmetic expression '{expression}'"


@tool
def get_weather(city: str) -> str:
    """Looks up weather information for a given city.

    Returns the temperature in Celsius and current weather condition.
    This is a demo weather source with fixed data for Lahore, Karachi,
    and Islamabad. Use this tool for weather or temperature questions.
    """
    data = _FAKE_WEATHER_DB.get(city.strip().lower())

    if data is None:
        return f"ERROR: no weather data available for city '{city}'"

    return json.dumps(data)


@tool
def get_product_price(product: str) -> str:
    """Looks up a product's price and specifications from the local
    products.json database.

    Use this tool when the user asks about a product's price,
    specifications, or wants to compare products. The input should
    be the product name, such as 'Laptop A'.
    """
    try:
        with open(_PRODUCTS_DB_PATH, encoding="utf-8") as f:
            db = json.load(f)

    except FileNotFoundError:
        return (
            f"ERROR: Product database file "
            f"'{_PRODUCTS_DB_PATH}' was not found."
        )

    except json.JSONDecodeError:
        return (
            f"ERROR: Product database file "
            f"'{_PRODUCTS_DB_PATH}' contains invalid JSON."
        )

    clean_product = product.strip().strip(".,'\"").lower()
    entry = db.get(clean_product)

    if entry is None:
        return f"ERROR: no product found named '{product}'"

    return json.dumps(entry)


@tool
def failing_product_lookup(product: str) -> str:
    """Deliberately raises an error to demonstrate LangChain tool error handling."""
    raise RuntimeError(
        f"Simulated database failure while looking up '{product}'"
    )


# -------------------------------------------------------------------
# Registration & Display
# -------------------------------------------------------------------

# Normal agent tools
TOOLS = [
    calculator,
    get_weather,
    get_product_price,
]

# Separate tool used only for the error-handling experiment
failing_tools = [
    failing_product_lookup,
]


# -------------------------------------------------------------------
# Display Registered Tools
# -------------------------------------------------------------------

print("Normal agent tools:")

for tool_instance in TOOLS:
    print(
        f"- {tool_instance.name}: "
        f"{tool_instance.description[:70]}..."
    )


print("\nFailure-testing tool:")

for tool_instance in failing_tools:
    print(
        f"- {tool_instance.name}: "
        f"{tool_instance.description[:70]}..."
    )

Normal agent tools:
- calculator: Evaluates a basic arithmetic expression and returns the numeric result...
- get_weather: Looks up weather information for a given city.

    Returns the temper...
- get_product_price: Looks up a product's price and specifications from the local
    produ...

Failure-testing tool:
- failing_product_lookup: Deliberately raises an error to demonstrate LangChain tool error handl...


### How Tool Docstrings Function as Part of the Prompt

The `@tool` decorator does more than register a Python function. LangChain uses the function's name, type hints, and docstring to build the tool schema that is provided to the model. The docstring becomes the tool's description, while the function signature and type annotations are used to describe the tool's input parameters.

This means the docstring effectively acts as part of the tool-selection instructions given to the model. A clear description helps the model understand when the tool should be used and what kind of input it expects. A vague or incomplete description can lead to incorrect or missed tool calls, similar to the problems we could encounter in Day 1 when manually defining tool descriptions.

Therefore, LangChain does not eliminate the need for good tool descriptions; it moves the responsibility from manually written JSON schemas to the function's name, signature, type hints, and docstring.


## Task 3: Build an Agent with `create_tool_calling_agent` / `AgentExecutor`


In [ ]:
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

In [7]:
agent_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a helpful assistant with access to several tools.

Use the available tools whenever they are needed to answer the user's
question accurately. Do not guess information that can be obtained
from a tool.

Available tools include:
- calculator: performs arithmetic calculations
- get_weather: looks up weather information
- get_product_price: looks up product prices and specifications

After using the necessary tools, provide a clear final answer to the user."""
    ),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])


# -------------------------------------------------------------------
# Create the tool-calling agent
# -------------------------------------------------------------------

agent = create_tool_calling_agent(
    llm,
    TOOLS,
    agent_prompt
)


# -------------------------------------------------------------------
# Wrap the agent in AgentExecutor
# -------------------------------------------------------------------

agent_executor = AgentExecutor(
    agent=agent,
    tools=TOOLS,
    verbose=True
)


# -------------------------------------------------------------------
# Invoke the agent with a multi-tool task
# -------------------------------------------------------------------

trace_result = agent_executor.invoke({
    "input": (
        "What's the weather in Lahore, and what's the price of "
        "Laptop A? Tell me both."
    )
})


# -------------------------------------------------------------------
# Display final answer
# -------------------------------------------------------------------

print("\n=== FINAL OUTPUT ===")
print(trace_result["output"])



> Entering new AgentExecutor chain...

Invoking: `get_weather` with `{'city': 'Lahore'}`


{"temp_c": 41, "condition": "sunny"}
Invoking: `get_product_price` with `{'product': 'Laptop A'}`


{"price": 899.0, "specs": "16GB RAM, 512GB SSD, mid-range CPU"}The weather in Lahore is sunny with a temperature of 41 degrees Celsius. The price of Laptop A is $899, and it comes with 16GB RAM, 512GB SSD, and a mid-range CPU.

> Finished chain.

=== FINAL OUTPUT ===
The weather in Lahore is sunny with a temperature of 41 degrees Celsius. The price of Laptop A is $899, and it comes with 16GB RAM, 512GB SSD, and a mid-range CPU.


### Annotating the Trace

The following annotation interprets the execution trace produced by `AgentExecutor`:

**[DECISION]**

The agent determines that the request requires both weather information and product information.

**[ACT]**

Invokes `get_weather("Lahore")`.

**[OBSERVE]**

Receives `41°C` and `sunny`.

**[ACT]**

Invokes `get_product_price("Laptop A")`.

**[OBSERVE]**

Receives Laptop A's price and specifications.

**[FINAL]**

Combines the tool results into the final response.

With `verbose=True`, `AgentExecutor` prints the tool-execution trace using LangChain's own output format rather than the explicit `[REASON]`, `[ACT]`, and `[OBSERVE]` labels used in the Day 1 raw-Python agent.

* `> Entering new AgentExecutor chain...` — indicates that the agent execution has started.
* `Invoking: get_weather with {'city': 'Lahore'}` — shows that the model selected the weather tool and supplied its arguments. This corresponds to the **Act** step.
* `{"temp_c": 41, "condition": "sunny"}` — shows the result returned by the weather tool. This corresponds to the **Observe** step.
* `Invoking: get_product_price with {'product': 'Laptop A'}` — shows the model selecting the product tool and providing the product argument. This is another **Act** step.
* The returned JSON containing Laptop A's price and specifications is the corresponding **Observation**.
* The final natural-language response is produced after the required tool calls have completed.
* `> Finished chain.` indicates that the `AgentExecutor` has completed the run.

The model's internal reasoning is **not displayed** as a separate step in this trace. Unlike the Day 1 implementation, where we explicitly logged events such as `[REASON] Model chose tool`, LangChain's verbose output primarily exposes the tool invocations, tool results, and final execution status. The `[DECISION]` label above is therefore our interpretation of what happened, not a line printed by LangChain.

### Compared with Day 1's Raw-Python Agent

The underlying process is similar: the model determines whether tools are needed, a tool is executed, its result is returned to the agent, and the process continues until a final answer can be produced. The major difference is that Day 1 required us to implement this orchestration loop manually, while `AgentExecutor` manages the execution process for us.

Several implementation details are now hidden behind LangChain's abstractions. The exact API request and response structures, tool-call message objects, and loop-control logic are not displayed by default. Configuration such as iteration limits and tool-error handling can be controlled through `AgentExecutor` parameters rather than through a manually written `while` loop. The `agent_scratchpad` is also managed as part of the agent execution process rather than being manually maintained in the same way as Day 1's message list.

This demonstrates an important framework trade-off: **LangChain reduces repetitive implementation work and provides useful abstractions, but some underlying execution details become less visible and require framework-specific knowledge when debugging.**


## Task 4: Add Memory

Using `RunnableWithMessageHistory` to manage conversation history, the agent handles a 3-turn conversation where later questions depend on earlier context. This provides the modern LangChain approach to maintaining message history across separate agent invocations.



In [8]:
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

In [9]:
# 1. Prompt template with chat history and agent scratchpad

memory_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a helpful shopping assistant with access to tools.
Use the tools when needed to answer questions accurately.
Remember information from earlier turns in the conversation."""
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])


# 2. Agent and Executor setup

memory_agent = create_tool_calling_agent(
    llm,
    TOOLS,
    memory_prompt
)

memory_executor = AgentExecutor(
    agent=memory_agent,
    tools=TOOLS,
    verbose=True
)


# 3. Session store and retrieval function

_session_store = {}


def get_session_history(session_id: str):
    if session_id not in _session_store:
        _session_store[session_id] = InMemoryChatMessageHistory()

    return _session_store[session_id]


# 4. Wrap the agent with RunnableWithMessageHistory

agent_with_memory = RunnableWithMessageHistory(
    memory_executor,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)


# Clear session store before running the test
# to prevent previous conversations from affecting the results

_session_store.clear()

config = {
    "configurable": {
        "session_id": "demo-session-1"
    }
}


# -------------------------------------------------------------------
# 5. Execute 3-turn memory scenario
# -------------------------------------------------------------------

turn1 = agent_with_memory.invoke(
    {"input": "Find the price of Laptop A."},
    config=config
)

print("\nTurn 1:", turn1["output"])


turn2 = agent_with_memory.invoke(
    {"input": "Now compare it to Laptop B."},
    config=config
)

print("\nTurn 2:", turn2["output"])


turn3 = agent_with_memory.invoke(
    {"input": "Which one should I recommend to a budget-conscious client?"},
    config=config
)

print("\nTurn 3:", turn3["output"])



> Entering new AgentExecutor chain...

Invoking: `get_product_price` with `{'product': 'Laptop A'}`


{"price": 899.0, "specs": "16GB RAM, 512GB SSD, mid-range CPU"}The price of Laptop A is $899. It has 16GB of RAM, a 512GB SSD, and a mid-range CPU.

> Finished chain.

Turn 1: The price of Laptop A is $899. It has 16GB of RAM, a 512GB SSD, and a mid-range CPU.


> Entering new AgentExecutor chain...

Invoking: `get_product_price` with `{'product': 'Laptop B'}`


{"price": 459.0, "specs": "8GB RAM, 256GB SSD, entry-level CPU"}
Invoking: `get_product_price` with `{'product': 'Laptop A'}`


{"price": 899.0, "specs": "16GB RAM, 512GB SSD, mid-range CPU"}Laptop A costs $899 and has 16GB of RAM, a 512GB SSD, and a mid-range CPU. Laptop B costs $459 and has 8GB of RAM, a 256GB SSD, and an entry-level CPU. Laptop A has better specs, but Laptop B is more budget-friendly.

> Finished chain.

Turn 2: Laptop A costs $899 and has 16GB of RAM, a 512GB SSD, and a mid-range CPU. Laptop B costs $459 

### Results
LangChain memory preserves conversation context across multiple turns, allowing the agent to resolve references such as "it" and "which one" from earlier messages. For example, Turn 2 ("compare it to Laptop B") only makes sense if the model remembers that Turn 1 was about Laptop A. The `chat_history` populated automatically by `RunnableWithMessageHistory` carries the previous conversation into the next invocation, serving the same role as Day 1's growing `messages` list but without requiring us to append and maintain the messages manually.

The test also showed that conversation memory does not necessarily prevent repeated tool calls. In Turns 2 and 3, the agent retrieved information about Laptop A again even though it had already retrieved it earlier. This demonstrates that **conversation memory and tool/data retrieval are separate concerns**: memory provides the previous conversational context, while the agent may still call a tool when it needs product information.


## Task 5: Structured Output & Error Handling

### Structured Output

The structured-output section uses LangChain's `with_structured_output()` with a Pydantic `Recommendation` model. The agent's product lookup tool is used to retrieve the real product information for Laptop A and Laptop B, and this information is then passed to a dedicated structured-output LCEL chain.

The structured-output chain compares the products according to the user's requirement and directly returns a validated `Recommendation` object containing the recommended product, price, and reason. In the test, the model returned `Laptop B`, a price of `459.0`, and the reason `"More affordable with decent specs"`, confirming that the structured output was successfully created as a `Recommendation` object.

This provides a predictable schema that can be consumed programmatically instead of relying on free-form text.


In [10]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

In [11]:
# -------------------------------------------------------------------
# 1. Define the required structured output
# -------------------------------------------------------------------

class Recommendation(BaseModel):
    """A structured product recommendation for a client."""

    recommended_product: str = Field(
        description="Name of the recommended product"
    )

    price: float = Field(
        description="Price of the recommended product"
    )

    reason: str = Field(
        description="Short reason for the recommendation"
    )


# -------------------------------------------------------------------
# 2. Create an LLM with native structured output
# -------------------------------------------------------------------

structured_llm = llm.with_structured_output(
    Recommendation
)


# -------------------------------------------------------------------
# 3. Create a prompt for the final structured answer
# -------------------------------------------------------------------

structured_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a product recommendation assistant.

Recommend the best product for a budget-conscious client.

Use ONLY the product information provided below.

Do not invent prices or product information.

Return the final answer using the required Recommendation
structured schema.
"""
    ),
    (
        "human",
        """Compare these products:

Product A:
{product_a}

Product B:
{product_b}

Recommend the better option for a budget-conscious client."""
    ),
])


# -------------------------------------------------------------------
# 4. Build the structured-output chain
# -------------------------------------------------------------------

structured_chain = (
    structured_prompt
    | structured_llm
)


# -------------------------------------------------------------------
# 5. Get REAL product data using the existing tool
# -------------------------------------------------------------------

product_a = get_product_price.invoke({
    "product": "Laptop A"
})

product_b = get_product_price.invoke({
    "product": "Laptop B"
})


print("=== PRODUCT DATA ===")
print("Laptop A:", product_a)
print("Laptop B:", product_b)


# -------------------------------------------------------------------
# 6. Generate the structured final answer
# -------------------------------------------------------------------

structured_result = structured_chain.invoke({
    "product_a": product_a,
    "product_b": product_b
})


# -------------------------------------------------------------------
# 7. Display structured output
# -------------------------------------------------------------------

print("\n=== STRUCTURED OUTPUT ===")

print(
    "Recommended Product:",
    structured_result.recommended_product
)

print(
    "Price:",
    structured_result.price
)

print(
    "Reason:",
    structured_result.reason
)

print("\n=== OUTPUT TYPE ===")
print(type(structured_result))

=== PRODUCT DATA ===
Laptop A: {"price": 899.0, "specs": "16GB RAM, 512GB SSD, mid-range CPU"}
Laptop B: {"price": 459.0, "specs": "8GB RAM, 256GB SSD, entry-level CPU"}

=== STRUCTURED OUTPUT ===
Recommended Product: Product B
Price: 459.0
Reason: More affordable option

=== OUTPUT TYPE ===
<class '__main__.Recommendation'>


### Error handling for tool failures

To make the failure test reproducible, the normal `get_product_price` tool reads the local JSON database without intentionally failing. Separate tools are used to demonstrate two failure-handling behaviors.

First, `broken_product_lookup` deliberately raises a `RuntimeError`. Because the exception is not caught, it propagates through `AgentExecutor` and stops the execution.

Second, `safe_product_lookup` catches the simulated failure using `try/except` and returns a controlled `TOOL_ERROR` message instead of raising the exception. The agent receives this message as a tool observation, follows the instruction not to invent product information, and returns a graceful response explaining that the lookup could not be completed.

This demonstrates that tool-level exception handling can make the agent resilient to failures. In this particular setup, the successful recovery is achieved by catching the exception inside the tool and returning a controlled error message.


In [12]:
# -------------------------------------------------------------------
# 1. Tool that intentionally crashes
# -------------------------------------------------------------------

@tool
def broken_product_lookup(product: str) -> str:
    """
    Deliberately raises an exception to demonstrate
    what happens when a tool failure is not handled.
    """
    raise RuntimeError(
        f"Simulated database failure while looking up '{product}'"
    )


# -------------------------------------------------------------------
# 2. Tool that handles its own failure gracefully
# -------------------------------------------------------------------

@tool
def safe_product_lookup(product: str) -> str:
    """
    Looks up a product but converts database failures
    into a controlled TOOL_ERROR message.
    """
    try:
        # -----------------------------------------------------------
        # Simulate a database failure.
        # In a real application, this could be:
        #
        #   data = read_database()
        #   result = data[product]
        #
        # -----------------------------------------------------------
        raise RuntimeError(
            f"Simulated database failure while looking up '{product}'"
        )

    except Exception as e:
        return (
            f"TOOL_ERROR: Product lookup failed for '{product}'. "
            f"Reason: {str(e)}. "
            "Do not invent or guess product information."
        )


# -------------------------------------------------------------------
# 3. Tools used for each experiment
# -------------------------------------------------------------------

broken_tools = [
    broken_product_lookup
]

safe_tools = [
    safe_product_lookup
]


# -------------------------------------------------------------------
# 4. Prompt for the failure-handling experiment
# -------------------------------------------------------------------

failure_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are testing tool error handling.

You have access to a product lookup tool.

Use the product lookup tool whenever the user asks
for product information or a product price.

The tool may fail.

If the tool returns a TOOL_ERROR:

1. Do not invent or guess product information.
2. Clearly explain that the lookup failed.
3. Explain that the product data source is unavailable.
4. Respond gracefully instead of terminating the conversation.
"""
    ),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])


# ===================================================================
# TEST 1: WITHOUT ERROR HANDLING
# ===================================================================

print("=" * 70)
print("TEST 1: WITHOUT ERROR HANDLING")
print("=" * 70)


# -------------------------------------------------------------------
# Create agent using the deliberately broken tool
# -------------------------------------------------------------------

broken_agent = create_tool_calling_agent(
    llm,
    broken_tools,
    failure_prompt
)


# -------------------------------------------------------------------
# Create executor WITHOUT error handling
# -------------------------------------------------------------------

broken_executor = AgentExecutor(
    agent=broken_agent,
    tools=broken_tools,
    verbose=True
)


# -------------------------------------------------------------------
# Run the failing tool
# -------------------------------------------------------------------

try:

    broken_result = broken_executor.invoke({
        "input": "Look up the price of Laptop C."
    })

    print("\n=== RESULT ===")
    print(broken_result["output"])

except Exception as e:

    print("\n[EXPECTED FAILURE]")
    print(f"Exception type: {type(e).__name__}")
    print(f"Exception message: {e}")


# ===================================================================
# TEST 2: WITH ERROR HANDLING
# ===================================================================

print("\n\n")
print("=" * 70)
print("TEST 2: WITH ERROR HANDLING")
print("=" * 70)


# -------------------------------------------------------------------
# Create agent using the safe tool
# -------------------------------------------------------------------

safe_agent = create_tool_calling_agent(
    llm,
    safe_tools,
    failure_prompt
)


# -------------------------------------------------------------------
# Create executor WITH error handling
# -------------------------------------------------------------------

safe_executor = AgentExecutor(
    agent=safe_agent,
    tools=safe_tools,
    verbose=True,
    handle_tool_errors=True,
    max_iterations=4
)


# -------------------------------------------------------------------
# Run the agent
# -------------------------------------------------------------------

try:

    safe_result = safe_executor.invoke({
        "input": "Look up the price of Laptop C."
    })

    print("\n=== FINAL OUTPUT (RESILIENT) ===")
    print(safe_result["output"])

except Exception as e:

    print("\n[UNEXPECTED FAILURE]")
    print(f"Exception type: {type(e).__name__}")
    print(f"Exception message: {e}")


# ===================================================================
# EXPECTED BEHAVIOR
# ===================================================================

print("\n\n")
print("=" * 70)
print("ERROR HANDLING SUMMARY")
print("=" * 70)

print("""
TEST 1:
Tool raises RuntimeError
        ↓
Exception propagates
        ↓
Agent execution stops

TEST 2:
Tool catches RuntimeError
        ↓
Returns TOOL_ERROR as tool observation
        ↓
Agent observes the error
        ↓
Agent does not invent product information
        ↓
Agent gives a graceful final response
""")

TEST 1: WITHOUT ERROR HANDLING


> Entering new AgentExecutor chain...

Invoking: `broken_product_lookup` with `{'product': 'Laptop C'}`



[EXPECTED FAILURE]
Exception type: RuntimeError
Exception message: Simulated database failure while looking up 'Laptop C'



TEST 2: WITH ERROR HANDLING


> Entering new AgentExecutor chain...

Invoking: `safe_product_lookup` with `{'product': 'Laptop C'}`


TOOL_ERROR: Product lookup failed for 'Laptop C'. Reason: Simulated database failure while looking up 'Laptop C'. Do not invent or guess product information.I apologize, but the product lookup for Laptop C has failed. The product data source is currently unavailable. I'm unable to provide the price of Laptop C at this time. If you'd like to try again later or look up a different product, I'd be happy to assist you.

> Finished chain.

=== FINAL OUTPUT (RESILIENT) ===
I apologize, but the product lookup for Laptop C has failed. The product data source is currently unavailable. I'm unable to prov

### What did I have to configure to make it graceful?

In my test, the deliberately broken tool raised a `RuntimeError`, which propagated through `AgentExecutor` and stopped the first run. For the resilient version, I added tool-level `try/except` handling inside `safe_product_lookup`, which catches the exception and returns a controlled `TOOL_ERROR` message instead of raising it. The agent receives this message as a tool observation, follows the instruction not to invent product information, and returns a graceful response. In this particular LangChain setup, the successful recovery therefore comes from handling the exception inside the tool rather than relying only on the executor-level `handle_tool_errors=True` option.

### What did LangChain make easier vs. Day 1's raw agent, and what "magic" did I notice?

Compared with my Day 1 raw-Python agent, LangChain removed much of the repetitive agent plumbing. The `@tool` decorator handled tool registration and schema generation from function signatures and docstrings, `AgentExecutor` managed the tool-calling loop, and `RunnableWithMessageHistory` handled conversation history across multiple turns. Structured output was also easier to implement with `with_structured_output()` and a Pydantic model. The main abstraction I noticed was that LangChain hides much of the message and tool-execution flow inside its components, so debugging can be less straightforward than in the raw-Python version where I could directly inspect and control each step. The error-handling experiment also showed that framework abstractions are convenient, but reliable recovery still requires understanding where exceptions are caught and how tool results are passed back to the agent.
